In [18]:
import pandas as pd

columns = [
    "id", "label", "statement", "subject", "speaker",
    "job_title", "state_info", "party_affiliation",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_fire_counts", "context"
]

train_df = pd.read_csv("data/train.tsv", sep = "\t", header=None, names = columns)
test_df = pd.read_csv("data/test.tsv", sep = "\t", header=None, names = columns, index_col=0)
cross_valid_df = pd.read_csv("data/valid.tsv", sep = "\t", header=None, names = columns, index_col=0)

print(train_df["label"].value_counts())

label
half-true      2114
false          1995
mostly-true    1962
true           1676
barely-true    1654
pants-fire      839
Name: count, dtype: int64


Columns names are not present with the dataset, so we create our own column names and assign it to our entries

In [20]:
label_map = {
    "pants-fire": 0, "false": 0, "barely-true": 0,
    "half-true": 1, "mostly-true": 1, "true": 1
}

train_df["binary_label"] = train_df["label"].map(label_map)
test_df["binary_label"] = test_df["label"].map(label_map)
cross_valid_df["binary_label"] = cross_valid_df["label"].map(label_map)

print(train_df["binary_label"].value_counts())

binary_label
1    5752
0    4488
Name: count, dtype: int64


We are collapsing 6 labels into 2 values- true or false, using .map

In [24]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample = train_df["statement"].iloc[0]
print(sample)

tokens = tokenizer(sample, padding = "max_length", truncation = True, max_length = 128)

print("\nKeys:", tokens.keys())
print("Input IDs:", tokens["input_ids"][:20])
print("Attention mask:", tokens["attention_mask"][:20])
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][:20]))


Says the Annies List political group supports third-trimester abortions on demand.

Keys: KeysView({'input_ids': [101, 2758, 1996, 8194, 2015, 2862, 2576, 2177, 6753, 2353, 1011, 12241, 20367, 11324, 2015, 2006, 5157, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1,

In [42]:
def tokenize(batch):
    return tokenizer(
        batch["statement"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_encodings = tokenize(train_df)
corss_valid_encodings   = tokenize(cross_valid_df)
test_encodings  = tokenize(test_df)

print("Input id:", train_encodings["input_ids"][1][:20]) 
print("Attention mask:", train_encodings["attention_mask"][1][:100]) 
print(len(train_encodings["input_ids"]))

Input id: [101, 2043, 2106, 1996, 6689, 1997, 5317, 2707, 1029, 2009, 2318, 2043, 3019, 3806, 2165, 2125, 2008, 2318, 2000, 4088]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
10240
